# Surgical Organ Classifier — PaliGemma 2 Fine-Tuning
Fine-tunes PaliGemma 2 on DSAD (Dresden Surgical Anatomy Dataset) for laparoscopic organ classification.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Add secrets (🔑): `HF_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY`
3. Upload DSAD to Google Drive at: `MyDrive/surgical-annotator/datasets/dsad/raw/Dresden Dataset/`
4. Run all cells top to bottom

In [ ]:
# 1 · Install dependencies
%%capture
!pip install -q \
    transformers>=4.40.0 peft accelerate datasets \
    Pillow numpy pycocotools scikit-learn matplotlib \
    tqdm huggingface_hub opencv-python-headless \
    kaggle kagglehub 'torchao>=0.16.0'
print('✓ Dependencies installed')

In [ ]:
# 2 · Verify GPU
import torch, os
assert torch.cuda.is_available(), '⚠ No GPU — switch runtime to T4'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# 3 · Load credentials + mount Drive
from google.colab import userdata, drive
import os

# Kaggle
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    print('✓ Kaggle credentials loaded')
except Exception as e:
    print(f'⚠ Kaggle: {e}')

# HuggingFace
try:
    import huggingface_hub
    huggingface_hub.login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('✓ HuggingFace credentials loaded')
except Exception as e:
    print(f'⚠ HuggingFace: {e}')

# Drive
drive.mount('/content/drive')

# Verify DSAD on Drive
DSAD_BASE = '/content/drive/MyDrive/surgical-annotator/datasets/dsad/raw/Dresden Dataset'
if os.path.exists(DSAD_BASE):
    print(f'✓ DSAD found on Drive')
    for item in os.listdir(DSAD_BASE):
        print(f'  {item}')
else:
    print('✗ DSAD not found — upload to Drive first')
    print(f'  Expected: {DSAD_BASE}')

In [ ]:
# 4 · Clone repo
import os, sys

REPO_URL = 'https://github.com/Vjp802/surgical-annotator'
REPO_DIR = '/content/surgical-annotator'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest...')
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

sys.path.insert(0, f'{REPO_DIR}/training')
print('✓ Training code available')

In [ ]:
# 5 · Configure training
import torch, os

DSAD_BASE    = '/content/drive/MyDrive/surgical-annotator/datasets/dsad/raw/Dresden Dataset'
REPO_DIR     = '/content/surgical-annotator'
MODEL_ID     = 'google/paligemma2-3b-pt-224'
OUTPUT_DIR   = '/content/drive/MyDrive/surgical-annotator/checkpoints'  # save to Drive!
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Dataset — reads directly from Drive, no local copy
DATASET_SPECS = [
    f'{DSAD_BASE}/annotations/instances_train.json:'
    f'{DSAD_BASE}/images/train:3.0',
]

# Hyperparameters — tuned for T4 (15GB VRAM)
EPOCHS       = 5
BATCH_SIZE   = 1      # must be 1 on T4 for 3B model
LR           = 2e-5
LORA_R       = 4      # low rank to save VRAM
LORA_ALPHA   = 8
LORA_DROPOUT = 0.05
PATIENCE     = 3
TRAIN_RATIO  = 0.8
VAL_RATIO    = 0.1
MAX_SAMPLES  = 2000   # use 2000 of 13195 images for free Colab
                      # set to 0 or remove flag for full dataset
BF16         = torch.cuda.is_available()

# Verify dataset paths
for spec in DATASET_SPECS:
    parts = spec.split(':')
    ok_json = '✓' if os.path.exists(parts[0]) else '✗ MISSING'
    ok_imgs = '✓' if os.path.isdir(parts[1])  else '✗ MISSING'
    print(f'JSON {ok_json}  {parts[0]}')
    print(f'imgs {ok_imgs}  {parts[1]}')

print(f'\nOutput     : {OUTPUT_DIR}')
print(f'Epochs     : {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LoRA r: {LORA_R}')
print(f'Max samples: {MAX_SAMPLES}  |  BF16: {BF16}')
print(f'\nEstimated time: ~{MAX_SAMPLES * EPOCHS * 2.5 / 3600:.1f} hours on T4')

In [ ]:
# 6 · Build training command
max_samples_flag = f'  --max_samples {MAX_SAMPLES}' if MAX_SAMPLES > 0 else ''

cmd = (
    f"python {REPO_DIR}/training/finetune.py"
    f"  --datasets {' '.join(DATASET_SPECS)}"
    f"  --output_dir {OUTPUT_DIR}"
    f"  --model_id {MODEL_ID}"
    f"  --epochs {EPOCHS}"
    f"  --batch_size {BATCH_SIZE}"
    f"  --learning_rate {LR}"
    f"  --lora_r {LORA_R}"
    f"  --lora_alpha {LORA_ALPHA}"
    f"  --lora_dropout {LORA_DROPOUT}"
    f"  --patience {PATIENCE}"
    f"  --train_ratio {TRAIN_RATIO}"
    f"  --val_ratio {VAL_RATIO}"
    + max_samples_flag
    + ('  --bf16' if BF16 else '')
)

print('Command:')
for part in cmd.split('  '):
    print(f'  {part}')

In [ ]:
# 7 · Run training
# Checkpoints save to Drive automatically — safe if session disconnects
import subprocess, os
from google.colab import userdata

env = os.environ.copy()
try:
    env['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
env['PYTHONPATH'] = f"{REPO_DIR}/training:{env.get('PYTHONPATH', '')}"
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

proc = subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, env=env,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

print(f'\n--- finetune.py exited with code {proc.returncode} ---')
if proc.returncode == 0:
    print('✓ Training complete! Checkpoint saved to Drive.')
else:
    print('Training FAILED. Check output above.')

In [ ]:
# 8 · Evaluate
CHECKPOINT   = f'{OUTPUT_DIR}/best_model'
EVAL_OUT_DIR = '/content/evaluation_results'

eval_cmd = (
    f"python {REPO_DIR}/training/evaluate.py"
    f"  --checkpoint {CHECKPOINT}"
    f"  --coco_exports {DSAD_BASE}/annotations/instances_train.json"
    f"  --images_dir {DSAD_BASE}/images/train"
    f"  --output_dir {EVAL_OUT_DIR}"
)

print('Running evaluation...')
!{eval_cmd}

# Show confusion matrix
cm_path = f'{EVAL_OUT_DIR}/confusion_matrix.png'
if os.path.exists(cm_path):
    from IPython.display import Image as IPyImage, display
    display(IPyImage(filename=cm_path))

# Show metrics
metrics_path = f'{EVAL_OUT_DIR}/metrics.json'
if os.path.exists(metrics_path):
    import json
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('\nMetrics:')
    print(json.dumps(metrics, indent=2))

In [ ]:
# 9 · Export production model to Drive
PRODUCTION_DIR = '/content/drive/MyDrive/surgical-annotator/production_model'

export_cmd = (
    f"python {REPO_DIR}/training/export_model.py"
    f"  --checkpoint {OUTPUT_DIR}/best_model"
    f"  --output {PRODUCTION_DIR}"
)

print('Exporting merged model to Drive...')
!{export_cmd}
print(f'\n✓ Production model saved to Drive')
print(f'\nTo use in the app:')
print(f'  1. Download production_model/ from Drive to training/production_model/')
print(f'  2. export VLM_BACKEND=finetuned')
print(f'  3. uvicorn app.main:app --reload')

In [ ]:
# 10 · Quick inference test on 5 DSAD images
import json, os
from PIL import Image
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
import torch

CHECKPOINT = f'{OUTPUT_DIR}/best_model'
PROMPT = '<image> Identify the highlighted surgical organ. Answer with the organ name only:'

print('Loading fine-tuned model...')
processor = AutoProcessor.from_pretrained(CHECKPOINT)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    CHECKPOINT,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
model.eval()
print('✓ Model loaded')

# Test on 5 sample images
img_dir = f'{DSAD_BASE}/images/train'
test_images = sorted(os.listdir(img_dir))[:5]

print('\nInference results:')
for fname in test_images:
    image = Image.open(f'{img_dir}/{fname}').convert('RGB')
    inputs = processor(images=image, text=PROMPT, return_tensors='pt').to(model.device)
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=8, do_sample=False)
    input_len = inputs['input_ids'].shape[1]
    pred = processor.decode(generated[0][input_len:], skip_special_tokens=True).strip()
    print(f'  {fname}  →  "{pred}"')

print('\n✓ Inference test complete')

---
## Notes

**T4 memory settings (already applied)**
- `BATCH_SIZE = 1` — required for 3B model on 15GB VRAM
- `LORA_R = 4` — low rank saves ~40% VRAM vs r=16
- Gradient checkpointing enabled in finetune.py
- `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`

**Checkpoints save to Drive**
- `OUTPUT_DIR` points to Drive so checkpoints survive session disconnects
- Best model saved at `checkpoints/best_model/`
- Latest checkpoint at `checkpoints/latest/`

**Scaling up**
- Free Colab: `MAX_SAMPLES = 2000`, 5 epochs (~2-3 hrs)
- Colab Pro T4: `MAX_SAMPLES = 0` (full 13K), 5 epochs (~6-8 hrs)
- Colab Pro+ A100: `MAX_SAMPLES = 0`, 10 epochs (~2-3 hrs)

**Adding surgeon corrections (Stage 2)**
Add your COCO export to DATASET_SPECS with weight 5.0:
```python
DATASET_SPECS = [
    f'{DSAD_BASE}/annotations/instances_train.json:{DSAD_BASE}/images/train:3.0',
    '/content/drive/MyDrive/surgical-annotator/exports/your_export.json:/path/to/images:5.0',
]
```

**Deploy to app**
```bash
export VLM_BACKEND=finetuned
export FINETUNED_MODEL_PATH=./training/production_model
uvicorn app.main:app --reload
```